# FIT5202 Data processing for Big data

##  Week 3 Lab: Parallel Search — From Plan to Partitions

Last week you discovered two things about Spark DataFrames: they are **immutable**, and they are **lazily evaluated**, a chain of `filter()`, `select()`, `orderBy()` calls does nothing until an **action** like `.show()` is called.

This week we go one level deeper and answer the question that was left at the end of Week 2:

> **Once an action triggers Spark's plan, how does that plan actually get executed across your machine?**

We'll answer it with one focused activity, a multi-condition *parallel search* over `bank_df`, and use it to see how Spark builds an execution plan, divides data into **partitions**, assigns **tasks** to process those partitions in parallel, and is affected by **data movement** and **uneven workloads**.

> **A quick note about this week's scope:** this week, we’re focusing on search operations. You might be wondering: what about `groupBy()` and joins? We’ll introduce those in following weeks, after you’ve developed a clear understanding of partitions, tasks, and data movement.

> **Before you begin:** This notebook assumes your Docker container is running, `resources/bank.csv` is available, and you're comfortable with `select()`, `filter()`, `orderBy()`, `limit()`, and method chaining from Week 2. If any of these operations still feel unfamiliar, it's worth a quick look back at last week's notebook first.

## Today's Plan

By the end of today's lab, you'll be able to:
1. predict what Spark will do with a chained query, and check that prediction against `explain()`;
2. explain the difference between the plan Spark builds and how Spark actually executes it;
3. inspect how a DataFrame is divided into partitions, including partition IDs and partition sizes,and compare partitioning strategies (round-robin, range, hash);
4. implement a multi-condition parallel search using both the DataFrame API and Spark SQL;
5. read a Spark job through the Spark UI;
6. identify a shuffle by finding an `Exchange` in the execution plan and shuffle evidence in the Spark UI;
7. recognise workload imbalance caused by data skew and explain how it can create a slow-running, or “straggler,” task;
8. explain why hash-partitioning a DataFrame by a column does **not** automatically allow Spark to skip partitions when searching that column.

>**Parts 1–10 form the guided core activity. The optional coding task in Part 5 and the Take-Home Practice provide additional independent practice to strengthen your understanding and support your assignment preparation**.

### Notebook Shortcuts
<font color='blue'>
    <strong>Notebook shortcuts you need to be familiar with and use frequently:</strong>

- Run your cells using SHIFT+ENTER (or "Run cell")
- Run the current cell and insert a new cell below: ALT+ENTER
- To see more commands, please click the "menu" option (e.g. "Insert", "Cell")
- To see more keyboard shortcuts, click the above "keyboard image" button. Use "Esc" to enter command mode. Then, you can use a command. Some of the popular shortcuts are
    - Basic navigation: enter, shift-enter, up/k, down/j
    - Saving the notebook: s
    - Cell creation: `a` = insert cell above, `b` = insert cell below
</font>

Let's get started.

## Table of Contents

1. [Week 2 Recap: Spark Builds a Plan](#part-1)
2. [Predict the Chained Search](#part-2)
3. [Meet the Partitions](#part-3)
4. [Compare Round-Robin, Range, and Hash Partitioning](#part-4)
5. [Parallel Search with the DataFrame API](#part-5)
6. [Repeat the Search with Spark SQL](#part-6)
7. [Interpret Jobs, Stages, and Tasks in the Spark UI](#part-7)
8. [Compare Filter and Repartition](#part-8)
9. [Compare Balanced and Skewed Workloads](#part-9)
10. [Final Challenge: Partitioning vs. Partition Pruning](#part-10)
11. [Final Reflection](#part-11)
12. [Take-Home Practice](#take-home)
13. [Stop Spark](#stop-spark)


<a id="part-1"></a>
# Part 1 — Week 2 Recap: Spark Builds a Plan

Let's start Spark the same way we did in Weeks 1 and 2, and reload `bank_df`.

In [ ]:
# Import SparkConf
from pyspark import SparkConf
from pyspark.sql import SparkSession

master = "local[*]"
app_name = "FIT5202-Week 3-Parallel Search"

spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("SparkSession created.")

In [ ]:
bank_df = spark.read.csv(
    "resources/bank.csv",
    header=True,
    inferSchema=True
)

bank_df.show(5)
print("Number of rows:", bank_df.count())
print("Number of partitions:", bank_df.rdd.getNumPartitions())

### Quick recap

Last week, you worked with several DataFrame operations. Today we only need five of them: `filter()`, `select()`, `orderBy()`, `limit()`, and method chain. Because the focus this week is **how Spark executes them**, not learning more spark DataFrame commands. Before we begin, let’s revisit two ideas from Week 2 that are more important than the syntax itself:

* **Immutability**: Every transformation returns a **new** DataFrame. It does not modify the original `bank_df`.
* **Lazy evaluation**: Transformations describe what should happen, but Spark does not process the data until an **action** such as `.show()`, `.count()`, or `.collect()` is called.

So, if Spark is not changing the original DataFrame and is not processing the data immediately, what is it doing?

It is building **a plan**. Let’s take a quick look at this in code.

In [ ]:
young_customers_df = bank_df.filter(bank_df.age < 30)
high_balance_df = bank_df.filter(bank_df.balance > 5000)

print("Original:", bank_df.count())
print("Young customers:", young_customers_df.count())
print("High balance:", high_balance_df.count())

Both DataFrames above were created from the *same* `bank_df`, and the original DataFrame remains unchanged.

Because transformations do not silently modify earlier DataFrames, Spark can examine the full sequence of operations, from the original data source to the final action, before deciding how to execute it:

> immutability -> transformation chain -> lazy evaluation -> **execution plan**

At the beginning of today’s applied class, the mini-lecture introduced the idea of a **Spark DAG**. We will now make this idea more concrete by following an executed query through its **job**, **stages**, and **tasks** in the Spark UI.

A **DAG (Directed Acyclic Graph)** represents the dependencies between Spark operations:
* When an **action** triggers execution, Spark organises the required work into a **job** containing one or more **stages**.
* A **shuffle** usually creates a boundary between stages.
* Within each stage, **tasks** process the individual data partitions.

As you explore the Spark UI, look for how these elements connect:
> **Action -> Job -> Stages -> Tasks**



<a id="part-2"></a>
# Part 2 — Predict the Chained Search

Take a look at the query below, but **don't** run it yet. 
>Note: `bank_df` doesn't have a `name` column, so we'll use `job` instead.

```python
top_balances_df = (
    bank_df
    .filter(bank_df.balance > 1000)
    .select("job", "balance")
    .orderBy("balance", ascending=False)
    .limit(5)
)
```

### Before You Run it

Write down your predictions. You can add them as comments in the next code cell:

1. Has Spark produced the five rows yet, just becasue we defined `top_balances_df`?
2. Does Spark have to execute `filter`, `select`, `orderBy`, and `limit` as four separate steps in exactly the order written?
3. How many times do you think Spark needs to scan the input data?
4. Which operations might require work to be *coordinated across partitions*, rather than allowing every partition to work independently?
5. What action could we call to trigger the computation?

Avoid answering only "which operation happens first?", that question  assumes there's one fixed, top-to-bottom order that matches the order you typed. Spark’s optimiser may combine, move, or reorganise operations, provided that the final result remains the same. 

In [ ]:
# Write your prediction

# YOUR ANSWER HERE

Now run the next cell and compare Spark’s plan with your predictions.

In [ ]:
top_balances_df = (
    bank_df
    .filter(bank_df.balance > 1000)
    .select("job", "balance")
    .orderBy("balance", ascending=False)
    .limit(5)
)

# Q1: has anything been computed yet? 
# Try checking the Spark UI Jobs tab before running the next cells.
print(type(top_balances_df))

Before we trigger the computation, let’s check the plan Spark has prepared:

In [ ]:
top_balances_df.explain(mode="formatted")

The key idea is:

> Transformations build a **logical description** of the requested result. Spark analyses this description and prepares a physical plan, but it does not process the actual data until we call an action.

This is a more precise way of saying, “Spark waits for an action.”

When you called `explain()`, Spark analysed and optimised the query so that it could show you the physical plan. However, it still did not run the query or produce the five rows. The actual data processing is still waiting for an action such as `.show()` or `.count()`.

Behind the scenes, a DataFrame contains a **logical plan** (a description of what result should be produced), when planning is triggered, Spark analyses and optimises that logical plan and turns it into a **physical plan**. the one you just saw in `explain()`. ([Spark `Dataset` documentation](https://spark.apache.org/docs/latest/api/java/org/apache/spark/sql/Dataset.html#explain()))

So far, Spark has planned the journey, but it hasn’t started moving through the data yet.

Now, call the action and compare the result to your predictions:

In [ ]:
top_balances_df.show()

Now, go back to your five predictions and check what you got right.

A few things to notice:
* Spark generally does **not** normally scan the entire dataset separately for every transformation. Several transformations can often be combined and processed as part of the same planned computation.
* Among these operations, `orderBy()` requires the most coordination across partitions. To find the true highest balances, Spark needs to compare values stored in *different* partitions.
* `limit(5)` does not simply return the first five rows Spark happens to encounter. Because it follows `orderBy()`, Spark must identify the five rows with the highest balances across the dataset.

For this query, a reasonable description is:

> One action normally triggers one planned computation, rather than four independent full scans.

However, treat this as something to check using **evidence**, rather than a rule that applies to every Spark query. The execution plan gives us some evidence already, and later we’ll confirm what happened using the Spark UI.

Let's get that evidence right now, with a small experiment of our own.


### From Written Order to Optimised Plan

So far you've been *told* that Spark does not always execute transformations in exactly the order they appear in your code. Its optimiser can reorganise or combine operations, as long as the final result stays the same. 

Let's check this directly, by writing the **same two conditions in a different order** and comparing what Spark actually plans to do with each.

In [ ]:
from pyspark.sql.functions import col

query_a = (
    bank_df
    .filter(col("balance") > 1000)
    .select("job", "balance")
)

query_b = (
    bank_df
    .select("job", "balance")
    .filter(col("balance") > 1000)
)

Both queries request the same result, but the transformations are written in a different order:
* Query A applies filter() before select().
* Query B applies select() before filter().

Will Spark preserve this difference, or will it optimise both queries into the same plan?

This time, we'll use `mode="extended"` instead of `mode="formatted"`.
The `formatted` mode is useful for quickly finding physical operators such as *Filter* and *Exchange*, but it does not show each stage of the logical planning process separately. Here, `mode="extended"` is more useful because it allows us to compare the different stages of Spark’s planning process.

* **Parsed Logical Plan**:  a fairly direct translation of the code you wrote.
* **Analyzed Logical Plan**: Spark resolves the column names and checks them against the DataFrame schema.
* **Optimized Logical Plan**: Catalyst applies optimisation rules and may reorganise, combine, or remove operations.
* **Physical Plan**: the concrete operators Spark will actually execute.

| Representation | Main question |
|---|---|
| Written DataFrame code | What did the programmer request? |
| Logical plan | What operations describe the requested result? |
| Optimised logical plan | How has Catalyst reorganised or combined those operations? |
| Physical plan | Which operators will Spark actually use? |
| Spark UI | What actually ran, as jobs, stages, and tasks? |

In [ ]:
print("===== QUERY A: filter, then select =====")
query_a.explain(mode="extended")

print("\n\n===== QUERY B: select, then filter =====")
query_b.explain(mode="extended")

### Discuss, using the output above as evidence

1. Is the order written by the programmer the same for `query_a` and `query_b`?
2. Compare the **Parsed Logical Plans**. Are they the same, or do they reflect the written order?
3. Compare the **Optimized Logical Plans**. Has Spark reorganised anything, and if so, are the two optimised plans now the same, or still different?
4. Compare the **Physical Plans**. Does Spark plan to execute the two queries differently?
5. Based on this evidence, does changing the written order necessarily change performance?

```text
Write transformations
        ↓
Spark records a logical plan
        ↓
Spark analyses and optimises the plan
        ↓
An action triggers data processing
        ↓
Tasks process partitions
        ↓
Results are returned or combined
```

This sequence highlights an important distinction. The first three steps are about **planning**. Spark can analyse and optimise a query before an action is called, as you saw when using `explain()`.

The remaining steps are about **execution**. The actual rows are not processed until an action triggers the planned computation.

For the rest of today’s lab, we’ll focus on this execution stage: *how the work is divided into partitions*, carried out by tasks, and combined to produce the result.

The main idea from this activity is:

> The order of your DataFrame methods describes the result you want, but it does not always determine the order in which Spark carries out the work. Spark makes that decision after its optimiser examines the complete query.

This is exactly the claim we introduced in Part 2 earlier part. Now, you have evidence for it, rather than just being told it.

<a id="part-3"></a>
# Part 3 — Meet the Partitions

A **partition** is a logical chunk of a distributed dataset that can be processed independently. 


Within a stage, Spark normally creates one **task** for each partition. This means that four partitions will normally produce four tasks. However, this does not mean that all four tasks will run at the same time. The number of tasks that can run concurrently depends on the number of processing cores available.

When we use `local[*]`, Spark can use the CPU cores available to the container. Different machines may therefore run a different number of tasks at the same time.

So, remember:
> Partitions determine how many tasks are created. Available cores determine how many tasks can run at the same time.

In Part 1, we only printed the number of partitions. That number is useful, but still quite abstract. Let’s make the partitions visible by checking:
1. Which row went to which partition?
2. How many rows are in each partition?

### Partition IDs

`spark_partition_id()` adds a column showing which partition each row currently belongs to. We then use `orderBy()` to arrange the displayed output by partition ID, making it easier to read.

In [ ]:
from pyspark.sql.functions import spark_partition_id

demo_df = spark.range(0, 20).repartition(4)

(
    demo_df
    .withColumn("partition_id", spark_partition_id())
    .orderBy("partition_id", "id")
    .show(20)
)

### Count the rows without collecting the whole dataset

`glom().collect()` (which you may have seen in some Spark material) is used to inspect partitions. It works for a tiny dataset, but it collects every row from every partition back to the driver. That is not a good habit for larger datasets.

Here, let's use`mapPartitionsWithIndex()`, it allows each partition to count its own rows. Only the small summary containing the partition IDs and row counts is returned to the driver.

In [ ]:
def get_partition_sizes(df):
    # Return a list of (partition_id, row_count) without collecting the actual rows.
    return (
        df.rdd
        .mapPartitionsWithIndex(
            lambda partition_id, rows: [
                (partition_id, sum(1 for _ in rows))
            ]
        )
        .collect()
    )

print(get_partition_sizes(demo_df))

### Creating a controlled, multi-partition version of `bank_df`

Depending on how your container reads `resources/bank.csv`, `bank_df` may load into a small number of partitions, sometimes just one. 

If that's the case, every experiment for the rest of today would show only a single task, which defeats the point of a *parallel* search lab. To avoid this, we’ll create a controlled version of bank_df with four partitions and use it throughout the rest of today’s lab.

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"> <strong style="color:#006DAE">Predict, then check:</strong> We’re about to distribute approximately 11,000 rows across four partitions using <b>repartition(4)</b>. This uses round-robin partitioning, which we’ll examine more closely in Part 4. Do you expect the four partitions to contain exactly the same number of rows? Why or why not? </div>

In [ ]:
parallel_bank_df = bank_df.repartition(4).cache()
parallel_bank_df.count() #materialise the cache now, before we start timing/inspecting anything

print("Number of partitions:", parallel_bank_df.rdd.getNumPartitions())
print(get_partition_sizes(parallel_bank_df))

Let’s break down what this cell did:

* `repartition(4)` redistributes the rows across four partitions. This makes parallel execution easier to observe with our small teaching dataset. With larger datasets, Spark normally determines the initial partitions from factors such as input splits and configuration settings.
* `.cache()` asks Spark to keep the resulting DataFrame in memory after it has been computed. This prevents the later activities from repeatedly rebuilding it from the CSV file.
* Calling `.count()` immediately afterwards is a deliberate **setup action**. It forces Spark to create `parallel_bank_df` and populate the cache now, before we begin inspecting or comparing different operations.

From this point onwards, we'll use `parallel_bank_df` instead of `bank_df`.

>**Keep this in mind**: because `parallel_bank_df` was created using `.repartition(4)`,so its lineage already contains an `Exchange`. As a result, calling `.explain()` on a DataFrame derived from `parallel_bank_df` may show this original Exchange, even if the new operation does not introduce another shuffle.
This will be important in Part 8. We’ll check whether an operation adds a new ``Exchange``, rather than simply looking for the word Exchange anywhere in the plan.

<a id="part-4"></a>
# Part 4 — Compare Round-Robin, Range, and Hash Partitioning

Partitions divide the data into units that Spark tasks can process. How the rows are distributed therefore affects how the work is distributed.

Let’s compare three common partitioning strategies:

* **Round-robin partitioning**: aims to distribute rows *approximately* evenly, without considering the values in a particular column. We request it using `.repartition(n)`. It does **not** guarantee every partition ends up with an identical workload, equal row counts can still involve different processing costs (e.g. some rows being more expensive to evaluate than others).
* **Range partitioning**: assigns different *ranges* of the selected key to different partitions. We request it using `.repartitionByRange(n, column)`. It arranges key ranges across partitions, but it does **not** guarantee fully sorted output and should not be treated as a replacement for `orderBy()`.
* **Hash partitioning**: Spark applies a hash function to the selected key and uses the result to choose a partition. We request it using `.repartition(n, column)`. Rows with the same key are always directed to the *same* partition, but different keys may also share a partition.

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"> <strong style="color:#006DAE">Predict, then check:</strong> We’ll distribute the same 10 rows across four partitions using each strategy. Which strategy do you expect to produce the most even partition sizes? Which strategy keeps equal keys together? Which strategy divides the data into meaningful key ranges? </div>

In [ ]:
demo_data = [
    (1, "Aaditya", "A"),
    (2, "Chinnavit", "B"),
    (3, "Neha", "A"),
    (4, "Huashun", "C"),
    (5, "Mohammad", "B"),
    (6, "Prajwol", "A"),
    (7, "Paras", "C"),
    (8, "Tooba", "B"),
    (9, "David", "A"),
    (10, "Cheng", "C"),
]

demo_df = spark.createDataFrame(demo_data, ["id", "name", "category"])

# Round-robin partitioning: 4 partitions, no particular column
demo_round = demo_df.repartition(4)

# Range partitioning: 4 partitions based on ranges of "name"
demo_range = demo_df.repartitionByRange(4, "name")

# Hash partitioning: 4 partitions, based on "category". 
# Note "category" repeats, (A appears 4 times, B and C 3 times each)
# unlike "id" which is unique per row and so could never actually show two rows sharing a key.
demo_hash = demo_df.repartition(4, "category")

print("----- ROUND ROBIN sizes -----")
print(get_partition_sizes(demo_round))
print("----- RANGE sizes -----")
print(get_partition_sizes(demo_range))
print("----- HASH sizes -----")
print(get_partition_sizes(demo_hash))

Partition *sizes* help us compare how evenly the rows were distributed. However, the sizes alone cannot show:
* whether range partitioning created meaningful ranges; or
* whether hash partitioning kept equal keys together.

For that, we need to see *which rows* landed in which partition.

In [ ]:
def show_partition_contents(df, columns):
    (
        df
        .withColumn("partition_id", spark_partition_id())
        .select("partition_id", *columns)
        .orderBy("partition_id", *columns)
        .show(truncate=False)
    )

print("===== RANGE partitioning: contents by name =====")
show_partition_contents(demo_range, ["name"])

print("===== HASH partitioning: contents by category =====")
show_partition_contents(demo_hash, ["category"])

Now compare the output with your predictions.

* For the **range-partitioned** DataFrame, each partition should contain a different alphabetical range of names. Earlier names should appear in the lower-numbered partitions, while later names should appear in the higher-numbered partitions.
>The `orderBy()` inside our helper function only makes the output easier to read. It is not part of `repartitionByRange()` itself and should not be taken as evidence that range partitioning produces fully sorted output.
* For the **hash-partitioned** DataFrame, all rows with `category == "A"` should appear in the *same* partition. You may also find that `"B"` and `"C"` share a partition. This demonstrates both parts of the definition:
    * Equal keys are kept together.
    * Different keys may still be placed in the same partition.

This is also why we switched the hash example to `"category"` rather than `"id"`: `id` is unique per row, so using it would not allow us to test whether repeated keys are kept together.

### Applying hash partitioning to `bank_df`

Let’s now apply hash partitioning to the larger bank dataset using the `education` column:

```python
hash_df = parallel_bank_df.repartition(4, "education")
```
Notice that we explicitly requested four partitions.

Without an explicit number, Spark uses its configured shuffle-partition count, often 200 by default. For this small teaching dataset, that could create many mostly empty partitions and make the output difficult to interpret.

In [ ]:
hash_df = parallel_bank_df.repartition(4, "education")
print(get_partition_sizes(hash_df))

Look at the partition sizes and consider:

1. Are the four partitions evenly sized?
2. Why might hashing `education` produce uneven partition sizes?
3. What would happen if one education category appeared much more frequently than all the others?

>(Extra practice: Take-Home Exercise 1 allows you to compare all three partitioning strategies using the full bank dataset. This will give you more practice interpreting how each strategy distributes the data.)

<a id="part-5"></a>
# Part 5 — Parallel Search with the DataFrame API

Now let's put partitions to work by running a real **multi-condition search**. 

Filtering is a useful example of parallel processing because each partition can usually evaluate the condition independently. Unlike operations such as sorting or aggregation, a filter does not normally need to compare rows from different partitions.

We'll search `parallel_bank_df` for customers who are:
* married,
* younger than 30,
* educated to primary or secondary level, and
* have a balance between 1000 and 2000 (inclusive).

In [ ]:
dataframe_result = (
    parallel_bank_df
    .filter(
        (parallel_bank_df.balance.between(1000, 2000)) &
        (parallel_bank_df.education.isin("primary", "secondary")) &
        (parallel_bank_df.age < 30) &
        (parallel_bank_df.marital == "married")
    )
    .select("age", "education", "marital", "balance")
)

# Note: this is the number of partitions in the RESULT DataFrame, 
# not the number of partitions that actually contained a matching row, getNumPartitions() can't tell you that.
print("Number of result partitions:", dataframe_result.rdd.getNumPartitions())
dataframe_result.show()

We called only one action, `.show()`, in the cell above. Remember that each action may trigger a new Spark job unless the result has already been cached. (`bank_df`'s multi-condition search happens to match fewer than 20 rows, so `.show()`'s default output already shows you every matching row, without needing a separate count.) If you're curious how many rows matched, `dataframe_result.count()` will tell you, just remember it triggers a job again.

The four conditions were combined inside one `filter()` call using `&`. We could also have written four chained `filter()` calls. Both forms are valid, and Catalyst can usually combine them into an equivalent predicate during optimisation.

Here, the combined form is mainly a readability choice: it presents the four conditions as one search.

### Investigate the plan

Use `explain(mode="formatted")` and what you learned about partitions to answer the following questions:

1. How many input partitions does `parallel_bank_df` have?
2. How many tasks would you expect for a stage that processes those partitions without a shuffle?
3. Does Spark need to examine every input partition?
4. Does this filter introduce a new `Exchange`?
5. How are the matching rows returned to `.show()`?

> At this point, make a prediction from the plan. We’ll verify the execution using the Spark UI in Part 7

In [ ]:
dataframe_result.explain(mode="formatted")

#### Lab Task 
<a class="anchor" id="lab-task-search"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Optional Lab Task: </strong> If you'd like more practice, design and run your own multi-condition search over <b>parallel_bank_df</b>, store it as <b>my_search_df</b>.
Your search must:

<ol>
    <li>use at least three conditions involving at least three different columns;</li>
    <li>select and display a sensible set of result columns; and</li>
    <li>print the number of partitions in the result DataFrame.</li>
</ol>

This is optional, `dataframe_result` above has already walked you through the pattern once, you may continue to Part 6 and spend more time on the Spark UI activities.

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [ ]:
# Optional Lab Task

# YOUR ANSWER HERE


<a id="part-6"></a>
# Part 6 — Repeat the Search with Spark SQL

The DataFrame API is not the only way to express a search in Spark. We can also write the same query using Spark SQL.

Both interfaces use the same underlying Spark SQL execution engine. This means that although the syntax is different, Spark can analyse and execute both queries using the same planning machinery. ([PySpark documentation](https://spark.apache.org/docs/latest/api/python/index.html))

To query a DataFrame with SQL, we first register `parallel_bank_df` as a temporary view.

In [ ]:
parallel_bank_df.createOrReplaceTempView("bank")

sql_result = spark.sql('''
    SELECT age, education, marital, balance
    FROM bank
    WHERE balance BETWEEN 1000 AND 2000
      AND education IN ('primary', 'secondary')
      AND age < 30
      AND marital = 'married'
''')

sql_result.show()

The `sql_result` should show the exact same rows as `dataframe_result` did, since both queries describe the same search conditions.

### Comparing the two approaches

Now compare the DataFrame and SQL versions using their formatted physical plans.

In [ ]:
print("----- DATAFRAME PLAN -----")
dataframe_result.explain(mode="formatted")

print("\n----- SQL PLAN -----")
sql_result.explain(mode="formatted")

### Discuss

1. Do both approaches return the same records?
2. Can you locate `Filter` and `Scan` (or `Relation`) in both plans?
3. The two approaches can produce equivalent, or very similar, optimised physical plans, but don't assume they'll always be *character-for-character identical*. Where (if anywhere) do the two plans above differ?
4. Does using SQL change the way Spark processes the partitions?
5. Which interface, DataFrame or SQL, do you find easier to read for this search?

**Main idea:** The DataFrame API and Spark SQL are two different ways to express a query. Underneath, Spark still builds a plan and executes it using the same engine and the same partitioned data.

<a id="part-7"></a>
# Part 7 — Interpret Jobs, Stages, and Tasks in the Spark UI

So far, `explain()` has shown you the *plan*. Now we’ll use the Spark UI to see what happened when an action actually triggered the plan.Open the Spark UI at: **http://localhost:4040**

As we learnt, when you call an action such as `.count()` or `.show()`, Spark turns the planned computation into executable work.

A useful starting relationship is:
> **Action -> Job -> Stage(s) -> Tasks**
- An **action** asks Spark to produce a result.
- A **job** represents the work triggered by that action.
- A job is divided into one or more **stages**.
- Within each stage, **tasks** process individual partitions.

Spark can keep operations in the same stage when partitions can be processed independently. When data must move between partitions, Spark performs a **shuffle**, which usually creates a boundary between stages.

### The Main Spark UI Views

The Spark UI provides several views of the same execution:

| Spark UI view | What to look for |
|---|---|
| **Jobs** | Which action triggered the work, and how many stages did it create?  |
| **SQL / DataFrame** | Which physical operators were executed? |
| **Stages** | Where was the work divided, and was shuffle involved? |
| **Tasks** | How many tasks ran, and did they receive similar workloads? |
| **Executors** | Which executor resources carried out the tasks? |

For today’s experiments, we’ll mainly use the **Jobs**, **SQL / DataFrame**, and **Stages** views.

### How to Follow an Experiment Through the UI

After running an experiment:
1. Open the **Jobs** tab.
2. Find the job created by the experiment.
3. Open the job and identify how many stages it contains.
4. Open the relevant stage.
5. Check how many tasks ran.
6. Look for shuffle read or shuffle write, if present.
7. Compare the task durations and input sizes.
8. Use the **SQL / DataFrame** tab to connect the executed operators with the physical plan shown by `explain()`.

When inspecting a stage, focus on:
- the number of tasks;
- task duration;
- shuffle read or shuffle write, when present; and
- whether one task appears to have much more work than the others.

These observations help us answer questions such as:
- Did Spark process several partitions in parallel?
- Did the operation require a shuffle?
- Did the shuffle create another stage?
- Was the work distributed reasonably evenly?
- Is there evidence that one task processed much more data than the others?

> **A small UI note:** The exact labels and metrics may vary slightly between Spark versions. Some input or shuffle metrics may also appear as zero when cached data is used or when a particular stage does not read directly from the original source. If this happens, use the task count, stage structure, physical plan, and partition sizes together as your evidence.

### Label Each Experiment

Several Spark jobs may appear in the UI, including jobs created by earlier setup actions. To make today’s experiments easier to find, we’ll give each one a description before triggering its action:

```python
sc.setJobDescription("Experiment A - Parallel filter")
some_df.count()
sc.setJobDescription(None) # reset, so later jobs aren't mislabelled
```

### For Each Experiment, Record Three Things

For each experiment in the next section, record:
1. **Prediction**: What do you expect before running the action?
2. **Evidence**: What do `explain()` and the Spark UI show?
3. **Explanation**: How does the evidence connect to partitions, tasks, stages, shuffle, or skew?

**Part 8 Experiment A** will guide you through this process step by step. Once you are familiar with the workflow, the later experiments will ask you to investigate more independently.

> About `.count()`: Spark may use a small final stage to combine the partial counts produced by different partitions into one result. When comparing experiments, focus mainly on the stages that process or redistribute the main dataset, rather than treating this small final combination stage as the central part of the experiment.

<a id="part-8"></a>
# Part 8 — Compare Filter and Repartition

Some operations can be done **entirely within a partition**. Others require Spark to move rows *between* partitions. 

This movement of data is called a **shuffle**. In a physical plan, a shuffle normally appears as an `Exchange`. In the Spark UI, it is reflected in the stage structure and may also appear through shuffle read and write metrics.

We'll compare two operations directly, a `filter()`, and a `repartition()` on a key column.

Both experiments use `.count()` so that they trigger comparable actions.

### Experiment A: Filter

A filter can normally evaluate each row inside its existing partition. It therefore should not add a new shuffle.

In [ ]:
sc.setJobDescription("Experiment A - Parallel filter")

search_result_df = parallel_bank_df.filter(parallel_bank_df.balance > 1000)
search_result_df.explain(mode="formatted")
search_result_df.count()

sc.setJobDescription(None)

Look near the top of the physical plan. You should see `Filter` operating over `InMemoryTableScan`, **without** a new `Exchange` introduced by the filter.

You'll still see an `Exchange` further down, but look carefully at where it appears. The `Exchange` uses:
```text
RoundRobinPartitioning(4), REPARTITION_BY_NUM
```
It appears inside the block describing how `parallel_bank_df` was originally cached. That came from the earlier `repartition(4)` used to create the controlled dataset. The question here is whether the **new filter operation** added another `Exchange`. And we can see that, the **filter** does not add a new `Exchange`. Each existing partition can be filtered independently without moving rows between partitions.

### Guided walkthrough: follow experiment A in the Spark UI

This is the one time today that we’ll work through the Spark UI step by step. Later experiments will reuse the same process.

#### 1. Find the labelled jobs
Open **http://localhost:4040** and select the **Jobs** tab. Find the jobs described as: `Experiment A - Parallel filter`

You may see **two jobs with this description**. This is expected.(see the `.count()` note above in Part 7.)

The job description is attached to both because both were triggered by the same experiment.Open the labelled job containing the stage with **four tasks**. This is the main stage that processed the four cached partitions.

#### 2. Check the number of tasks
Open the main stage and scroll to: **Summary Metrics for Completed Tasks**.Then find the **Tasks** table below it. Record:
- the number of completed tasks;
- the minimum, median, and maximum task durations; and
- whether the tasks processed similar amounts of input.

Compare the number of tasks with the number of partitions in `parallel_bank_df`.
> If the stage contains four tasks, this matches the four partitions created earlier. Within this stage, Spark created one task for each partition.

#### 3. Check whether the work was evenly distributed
Compare the task durations and input sizes. Ask:
- Are the durations reasonably similar?
- Did each task process a similar amount of input?
- Is one task much slower or larger than the others?

Small differences are normal. We are looking for a clear imbalance, such as one task taking substantially longer or processing substantially more data.

#### 4. Follow the query through the SQL / DataFrame view
Open the **SQL / DataFrame** tab and select the query for Experiment A. Find the `InMemoryTableScan` box in the diagram, that's the Spark UI's picture of the same physical plan `explain()` printed as text. Use it to check the following:
- `InMemoryTableScan` shows that Spark read parallel_bank_df from the cache.
- The output-row metric under `InMemoryTableScan` shows how many cached rows entered the query.
- The output-row metric under Filter shows how many rows satisfied the filter condition.
- The first `HashAggregate` produces one partial count from each partition.
- The following `Exchange` moves those small partial counts so they can be combined.
- The final `HashAggregate` produces the single result returned by `.count()`.
You may also see an earlier `Scan csv -> Exchange` above `InMemoryTableScan`. This describes how `parallel_bank_df` was originally created. The current experiment reads from the cache, so this earlier Exchange is not a new shuffle caused by the filter.

#### 5. Interpret the shuffle carefully
The filter itself does not require rows to move between partitions. Each partition can apply balance > 1000 independently.

However, `.count()` must return one final number. Spark therefore produces one partial count from each partition and performs a small shuffle to combine those partial results.The shuffle shown after the partial HashAggregate belongs to the count aggregation, not to the filter.

#### 6. Record your conclusion
For each experiment in the next section, record:
1. **Prediction**: What do you expect before running the action?
2. **Evidence**: What did the Spark UI and physical plan show?
3. **Explanation**: How do partitions, tasks, caching, filtering, and the final count explain the observed execution?

In [ ]:
## Write Your Prediction
## Your Answer Here

### Experiment B: Repartition

`repartition(4, "education")` asks Spark to redistribute the rows using a hash of the `education` value. Rows may need to move from their current partitions to new ones, so this operation requires a shuffle.

In [ ]:
sc.setJobDescription("Experiment B - Repartition by education")

education_partitioned_df = parallel_bank_df.repartition(4, "education")
education_partitioned_df.explain(mode="formatted")
education_partitioned_df.count()

sc.setJobDescription(None)

### Investigate experiment B

This time, the physical plan should show a **new `Exchange`** above the `InMemoryTableScan`.

The older `Exchange` inside the cached DataFrame’s lineage shows how `parallel_bank_df` was originally created. The new `Exchange` shows that `repartition(4, "education")` redistributes the cached rows again.

Now, **repeat the same six-step walkthrough from Experiment A yourself**, this time for the `Experiment B - Repartition by education` job.

### Compare Experiments A and B

1. Which experiment introduced a new `Exchange`?
2. Which column and partitioning strategy were used?
3. Why did Experiment B require rows to move between partitions?
4. How is this repartition shuffle different from the small shuffle used to combine partial counts in Experiment A?

<a id="part-9"></a>
# Part 9 — Compare Balanced and Skewed Workloads

Partitioning doesn't just decide *how many* pieces the data is split into, it also determines how much data—and therefore how much potential work—each task receives.

If one partition is much larger than the others, its task may take much longer to finish. This slow task is often called a **straggler**. Because a stage cannot finish until all its tasks are complete, one straggler can delay the entire stage.

To make this visible, we'll build two controlled synthetic datasets of the same size and hash-partition each by a different kind of key:

- a dataset hash-partitioned by the high-cardinality `id` column, which should produce an approximately balanced distribution; and
- a dataset hash-partitioned by a low-cardinality `key` column, where approximately 97% of rows have the value `"A"`.

**Cardinality** means the number of distinct values in a column:
- `id` has high cardinality because every row has a different value.
- `key` has low cardinality because it contains only `"A"`, `"B"`, `"C"`, and `"D"`.
> High cardinality does not guarantee perfectly equal partitions. However, hashing many distinct, well-distributed values usually produces a more balanced distribution than hashing only a few highly repeated values.

In [ ]:
from pyspark.sql.functions import col, when

ROWS = 1_000_000

# Create the skewed dataset
# Low-cardinality, heavily skewed key: about 97% of rows share the same value
skewed_df = spark.range(0, ROWS).withColumn(
    "key",
    when(col("id") % 100 < 97, "A")
    .when(col("id") % 100 < 98, "B")
    .when(col("id") % 100 < 99, "C")
    .otherwise("D")
)

# High-cardinality key: "id" has one million distinct values
# so later when hashing it should distribute the rows approximately evenly across partitions.
balanced_df = spark.range(0, ROWS)

Now hash-partition each dataset by its selected key. We’ll cache and materialise the results immediately, following the same pattern used for `parallel_bank_df` in Part 3.

This allows us to:
- perform each repartition shuffle once;
- reuse the resulting cached partitions; and
- label the actions whose Spark UI evidence we want to inspect.

In [ ]:
sc.setJobDescription("Balanced workload - repartition by id")
balanced_partitioned = balanced_df.repartition(4, "id").cache()
balanced_partitioned.count()
sc.setJobDescription(None)

sc.setJobDescription("Skewed workload - repartition by key")
skewed_partitioned = skewed_df.repartition(4, "key").cache()
skewed_partitioned.count()
sc.setJobDescription(None)

# Cheap: both DataFrames are already materialised and cached above,
# so this reads from the cache rather than repeating the shuffle.
print("Balanced partition sizes:", get_partition_sizes(balanced_partitioned))
print("Skewed partition sizes:  ", get_partition_sizes(skewed_partitioned))

Compare the partition sizes.

The balanced case should contain approximately similar numbers of rows in each partition.

In the skewed case, the partition receiving key `"A"` should contain most of the rows. 
> Four partitions do not guarantee four equal workloads. They only provide four possible units of work. The partitioning key determines how the rows are distributed among them.

Now open the Spark UI and select the **Stages** tab.In the **Description** column, find the stages labelled:
- `Balanced workload - repartition by id`
- `Skewed workload - repartition by key`

Several stages may share each label because the `.count()` action involves both the repartition shuffle and the final count aggregation.

For each workload, open the labelled stage that has:
- **four tasks**; and
- a non-zero **Shuffle Read**.

Then scroll down to its **Tasks** table.
This is the stage that reads and processes the four newly created partitions. Compare the task-level:
- **Shuffle Read Size / Records**;
- task durations; and
- distribution of work across the four tasks.

Use the output from `get_partition_sizes()` as your primary evidence of how many rows landed in each partition. Use the Spark UI task metrics as supporting evidence.

> Spark UI metrics may vary across environments. If some record or size values display `0`, the partition-size output still provides reliable evidence of whether the workload is balanced or skewed.


### Answer, using evidence
1. Do four partitions guarantee four equal workloads?
2. Which partition contains most of the rows in the skewed case?
3. Which task would you expect to take the longest, and why?
4. Can the stage finish before its slowest task is complete?
5. Why might adding more executor cores fail to produce a proportional speed-up when the data is heavily skewed?

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#006DAE">Connect back to real data:</strong> In <b>bank_df</b>, the <b>poutcome</b> column (outcome of the previous marketing campaign) contains only a few distinct values, and most rows are <b>"unknown"</b>. If you hash-partitioned <b>bank_df</b> by <b>poutcome</b>, what distribution might result from hash-partitioning by <b>poutcome</b>? How might this differ from hash-partitioning by a higher-cardinality column such as <b>balance</b>?
</div>

<a id="part-10"></a>
# Part 10 — Final Challenge: Partitioning vs. Partition Pruning


Hash partitioning places rows with the **same key** in the **same partition**. It is therefore tempting to conclude from that:
> *"so if I hash-partition by `job` and then filter for `job == 'management'`, Spark should only need to look at one partition, just like an index."*

That conclusion is worth testing rather than assuming.

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Final Challenge: </strong>

<ol>
    <li><b>Predict first, before writing any code:</b> will repartitioning <b>parallel_bank_df</b> by <b>job</b> allow Spark to inspect only one partition when later searching for <b>job == "management"</b>? Write your prediction and reasoning in a markdown cell.</li>
    <li>Repartition <b>parallel_bank_df</b> by <b>job</b> into 4 partitions, storing the result as <b>job_partitioned_df</b>. Cache the result and call <code>.count()</code> to materialise it.</li>
    <li>Build <b>management_df</b> by filtering <b>job_partitioned_df</b> for <b>job == "management"</b>. Call <b>get_partition_sizes()</b> on <b>management_df itself</b> (not on the unfiltered <b>job_partitioned_df</b>) to see exactly which partitions still contain a "management" row.</li>
    <li>Run the same filter again with a job description set, trigger it, and check the Spark UI: how many tasks ran in the stage that does the filtering?</li>
    <li>Explain, using that evidence, whether Spark actually skipped any partitions, or whether it still created a task for every partition, even the ones you now know (from step 3) contain no matching rows.</li>
</ol>
      
> **Why materialise before filtering (Step 2):**
>
> This experiment asks whether a filter can skip partitions that have **already been created**.
>
> Spark uses lazy evaluation, so defining `repartition()` does not immediately create the new partitions. If we add a filter before triggering an action, Spark may optimise the whole pipeline by applying the filter first and repartitioning only the remaining rows.
>
> That is normally a useful optimisation, but it would change what this experiment is testing.
>
> By caching the repartitioned DataFrame and calling `.count()` first, we force Spark to create and store the partitions before applying the filter. The later filter must therefore operate on these existing cached partitions.  
    
<strong>COMPLETE YOUR PREDICTION AND THE CODE BELOW.</strong>
</div>

</div>

**Your prediction (before coding):**

*YOUR ANSWER HERE*

In [ ]:
# Final Challenge — implementation

# 1. Repartition parallel_bank_df by "job" into 4 partitions, THEN .cache() and
#    .count() it immediately to materialise it before anything filters it further.
# YOUR ANSWER HERE


# 2. Build management_df = job_partitioned_df.filter(...), then call
#    get_partition_sizes(management_df) -- NOT get_partition_sizes(job_partitioned_df) --
#    to see exactly which partitions still contain "management" rows.
# YOUR ANSWER HERE


# 3. Re-run the filter with a job description set, trigger it, then check the Spark UI
# YOUR ANSWER HERE


Work through your prediction, evidence, and explanation *before* expanding the note below, the investigation is much more useful if you reach your own conclusion first!

<details>
<summary><b style="color:#FF5555">Click to check your conclusion after completing the investigation</b></summary>

> Hash partitioning places equal keys together, but it does not automatically give an in-memory DataFrame the metadata needed for partition pruning. Even if all `"management"` rows are located in one partition, Spark still creates a task for each input partition to evaluate the filter. 

This matters because it separates three ideas that can easily be confused:  
1. **Data distribution:** Hash partitioning decides which partition receives each row. Rows with the same key are placed in the same partition.
    
2. **Parallel scanning:** Spark can assign a task to each input partition, allowing the partitions to be scanned in parallel.

3. **Partition pruning:** Spark skips partitions that it knows cannot contain matching rows.
    Hash partitioning provides the first property: it keeps equal keys together. However, it does not automatically provide partition pruning.
    For example, all rows with `job == "admin."` may be stored in one partition, but Spark may still scan all four partitions when applying the filter. To skip the other partitions, Spark would need *additional metadata* that identifies which values can be found in each partition.

</details>

<a id="part-11"></a>
# Part 11 — Final Reflection

Today we moved from *what* Spark computes to *how* Spark computes it. Take a moment to reflect, in your own words:

> **Based on today's lab, how would you now explain to a Week 1 version of yourself what happens between writing `bank_df.filter(...)` and seeing results on screen?**

Some ideas that should show up somewhere in your answer:
- Transformations build a logical description of the requested result.
- Spark analyses and optimises the plan before executing it.
- An action triggers the actual data processing.
- The dependencies between operations can be represented as a DAG.
- Within a stage, partitions become units of work that are processed by tasks.
- Available executor cores determine how many tasks can run at the same time.
- The DataFrame API and Spark SQL can express the same query and produce equivalent plans.
- `filter()` can usually process existing partitions independently, while `repartition()` requires a shuffle.
- A stage must wait for its slowest task, which is why data skew matters.
- Hash partitioning keeps equal keys together, but does not automatically enable partition pruning.

💡There isn't a single correct answer. The goal is to connect today's evidence, plans and the Spark UI, to the underlying ideas, in your own words.

<a id="take-home"></a>
# Take-Home Practice

Today's lab focused on *how* Spark executes a search. The following two exercises give you a little more practice with the same ideas, using only what you've already learned today.

Work through them using `parallel_bank_df` (or `bank_df`, where noted).

### 1. Compare Three Partitioning Strategies

Create six partitions using:
- round-robin partitioning;
- range partitioning by `age`; and
- hash partitioning by `marital`.

Use `get_partition_sizes()` to compare the distributions.

**Question:** Which strategy produces the most even row counts for this dataset, and why?

In [ ]:
# YOUR ANSWER HERE

### 2. Spot the shuffle

Using `explain()`, build **two** queries over `parallel_bank_df`:

1. a query using only `filter()` and `select()`, which should add no new `Exchange`; and
2. a query using `repartition()` or `orderBy()`, which should add a new `Exchange`.

Remember that the lineage of `parallel_bank_df` may still contain the older `Exchange` used to create its four partitions. Focus on whether your new operation introduces another one.

**Question:** How do these two plans compare with Experiment A and Experiment B in Part 8?

In [ ]:
# YOUR ANSWER HERE

<a id="stop-spark"></a>
## Stop Spark

Run the next cell only when you have completely finished the notebook.

Stopping Spark releases the resources used by the active `SparkSession`.

In [ ]:
spark.stop()
print(
    "SparkSession stopped. Great work today ≧∇≦! You've gone from writing a chain of "
    "transformations to reading how Spark actually executes it across partitions. "
    "See you next week, when we add joins on top of this foundation."
)